In [14]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import pandas as pd
import numpy as np

# =====================================================================
# HÀM FEATURE ENGINEERING (ĐÃ ĐỒNG BỘ TÊN FEATURES VÀ BỎ QUA CHU KỲ 30)
# =====================================================================
def advanced_feature_engineering(main_df, oil_df, holidays_df, stores_df):
    # 1. Tạo bản sao dữ liệu
    df = main_df.copy()
    
    # 2. Đồng bộ định dạng thời gian
    df['date'] = pd.to_datetime(df['date'])
    oil_df['date'] = pd.to_datetime(oil_df['date'])
    holidays_df['date'] = pd.to_datetime(holidays_df['date'])
    
    # 3. Merge với bảng Cửa hàng và Giá dầu
    df = df.merge(stores_df, on='store_nbr', how='left')
    df = df.merge(oil_df, on='date', how='left')
    
    # 4. Trích xuất các đặc trưng thời gian cơ bản
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek  # 0: Thứ 2, 6: Chủ Nhật
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Đặc trưng ngày phát lương (wages) và cuối tháng
    df['is_payday'] = ((df['day'] == 15) | df['date'].dt.is_month_end).astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    
    # Tạm thời gán mặc định cho Easter Week nếu dữ liệu chưa tính toán
    df['is_easter_week'] = 0 
    
    # 5. Xử lý ngày lễ (Holidays)
    real_holidays = holidays_df[holidays_df['transferred'] == False]
    national_hols = real_holidays[real_holidays['locale'] == 'National'].drop_duplicates(subset=['date'])
    
    df = df.merge(national_hols[['date', 'type']], on='date', how='left')
    
    # Đổi tên và xử lý cột ngày lễ quốc gia thành is_real_holiday
    if 'type_y' in df.columns:
        df.rename(columns={'type_y': 'holiday_national_type'}, inplace=True)
    elif 'type' in df.columns:
        df.rename(columns={'type': 'holiday_national_type'}, inplace=True)
        
    if 'holiday_national_type' in df.columns:
        df['is_real_holiday'] = df['holiday_national_type'].notna().astype(int)
        df['holiday_national_type'] = df['holiday_national_type'].fillna('None')
    else:
        df['is_real_holiday'] = 0
        df['holiday_national_type'] = 'None'
        
    # Đổi tên loại cửa hàng cho đúng chuẩn store_type của bạn
    if 'type_x' in df.columns:
        df.rename(columns={'type_x': 'store_type'}, inplace=True)
        
    # 6. Tạo các biến Trễ (Lag) và biến Trượt (Rolling) - CHỈ LẤY CHU KỲ 7 VÀ 14 NGÀY
    df.sort_values(['store_nbr', 'family', 'date'], inplace=True)
    
    # Dịch chuyển gốc 16 ngày để dự báo tập test tương lai (Tránh rò rỉ dữ liệu)
    base_lag = 16 
    
    if 'sales' in df.columns:
        # Tạo 3 biến Lag chính ứng với tên bạn yêu cầu
        df['sales_lag_1'] = df.groupby(['store_nbr', 'family'])['sales'].shift(base_lag)      # Gốc lag 16
        df['sales_lag_7'] = df.groupby(['store_nbr', 'family'])['sales'].shift(base_lag + 1)  # Gốc lag 17
        df['sales_lag_14'] = df.groupby(['store_nbr', 'family'])['sales'].shift(base_lag + 2) # Gốc lag 18
        
        # Chỉ tạo Rolling Mean và Std cho chu kỳ 7 và 14 ngày (Bỏ hoàn toàn 30 ngày)
        for window in [7, 14]:
            df[f'rolling_mean_{window}'] = (
                df.groupby(['store_nbr', 'family'])['sales_lag_1']
                .transform(lambda x: x.rolling(window).mean())
            )
            df[f'rolling_std_{window}'] = (
                df.groupby(['store_nbr', 'family'])['sales_lag_1']
                .transform(lambda x: x.rolling(window).std())
            )

    # 7. Mã hóa tự động các cột dạng chữ thành số (Label Encoding) ngoại trừ cột date
    dynamic_categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if 'date' in dynamic_categorical_cols:
        dynamic_categorical_cols.remove('date')
        
    for col in dynamic_categorical_cols:
        df[col] = df[col].astype('category').cat.codes
            
    df = df.fillna(0)
    return df

In [17]:
import pandas as pd  # Thư viện dùng để xử lý dữ liệu dạng bảng (Dataframe)
import numpy as np   # Thư viện dùng để tính toán toán học và mảng số học

# ==================================================
# BƯỚC 1: ĐỌC TẤT CẢ CÁC BẢNG DATA GỐC VÀ SẠCH
# ==================================================
print("🔄 Đang đọc dữ liệu...")  # In thông báo trạng thái đang tải file

# Đọc file chứa thông tin phân loại và khu vực của các cửa hàng
stores_df = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\stores.csv")

# Đọc file lịch trình các ngày lễ, sự kiện diễn ra tại Ecuador
holidays_df = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\holidays_events.csv")

# Đọc file dữ liệu biến động giá dầu hàng ngày (đã xử lý ffill)
oil_df = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\oil.csv") 

# Đọc tập dữ liệu huấn luyện lịch sử (chứa đầy đủ sales và transactions)
train_data = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\train.csv")

# Đọc tập dữ liệu cần dự đoán trong tương lai (không có cột sales)
test_data = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\data\test.csv")

# Chuyển đổi cột 'date' của tập train từ chuỗi văn bản (str) sang dạng thời gian thực tế (datetime)
train_data['date'] = pd.to_datetime(train_data['date'])

# Chuyển đổi cột 'date' của tập test từ chuỗi văn bản (str) sang dạng thời gian thực tế (datetime)
test_data['date'] = pd.to_datetime(test_data['date'])

holidays_df['date'] = pd.to_datetime(holidays_df['date'])
oil_df['date']      = pd.to_datetime(oil_df['date'])

# ==================================================
# BƯỚC 2: GỘP TRAIN VÀ TEST ĐỂ TÍNH LAG/ROLLING
# ==================================================
print("🔗 Đang gộp tập dữ liệu Train và Test...")  # In thông báo gộp bảng

# Xếp chồng tập train và test nối liền nhau để có bệ đỡ lịch sử liên tục khi dịch chuỗi thời gian
full_data = pd.concat([train_data, test_data], ignore_index=True)


# ==================================================
# BƯỚC 3: ĐỊNH NGHĨA HÀM ADVANCED FEATURE ENGINEERING ĐẦY ĐỦ
# ==================================================
def advanced_feature_engineering(df_input, oil, holidays, stores):
    print("   ↳ 1. Sao lưu dữ liệu nền...")
    df_res = df_input.copy()
    
    # --- 1. GHÉP DỮ LIỆU CỬA HÀNG (STORES) ---
    print("   ↳ 2. Ghép thông tin vị trí & phân loại cửa hàng...")
    df_res = pd.merge(df_res, stores, on='store_nbr', how='left')
    df_res = df_res.rename(columns={'type': 'store_type'})  # Tránh trùng tên với cột type của bảng lễ
    
    # --- 2. XỬ LÝ VÀ GHÉP DỮ LIỆU NGÀY LỄ (HOLIDAYS) ---
    print("   ↳ 3. Xử lý bộ lọc ngày nghỉ lễ quốc gia thực tế...")
    holidays_temp = holidays.copy()
    holidays_temp['is_real_holiday'] = True
    holidays_temp.loc[(holidays_temp['type'] == 'Holiday') & (holidays_temp['transferred'] == True), 'is_real_holiday'] = False
    holidays_temp.loc[holidays_temp['type'] == 'Work Day', 'is_real_holiday'] = False
    
    # Lọc lấy các ngày lễ quy mô Toàn quốc (National) để tránh nhiễu cục bộ và loại bỏ trùng ngày
    holidays_clean = (
        holidays_temp[holidays_temp['locale'] == 'National']
        [['date', 'type', 'is_real_holiday']]
        .drop_duplicates(subset=['date'], keep='first')
    )
    holidays_clean = holidays_clean.rename(columns={'type': 'holiday_national_type'})
    
    # Tiến hành gộp thông tin ngày lễ vào bảng tổng
    df_res = pd.merge(df_res, holidays_clean, on='date', how='left')
    df_res['is_real_holiday'] = df_res['is_real_holiday'].fillna(False).astype(int)
    df_res['holiday_national_type'] = df_res['holiday_national_type'].fillna('None')
    
    # --- 3. GHÉP VÀ XỬ LÝ KHUYẾT GIÁ DẦU (OIL) ---
    print("   ↳ 4. Ghép chỉ số biến động giá dầu vĩ mô...")
    df_res = pd.merge(df_res, oil, on='date', how='left')
    df_res['dcoilwtico'] = df_res['dcoilwtico'].ffill()  # Forward fill cho các ngày nghỉ
    df_res['dcoilwtico'] = df_res['dcoilwtico'].bfill()  # Backward fill cho các ngày trống đầu chuỗi
    
    # --- 4. TRÍCH XUẤT ĐẶC TRƯNG CHU KỲ THỜI GIAN ---
    print("   ↳ 5. Trích xuất đặc trưng các thành phần thời gian...")
    df_res['year'] = df_res['date'].dt.year
    df_res['month'] = df_res['date'].dt.month
    df_res['day'] = df_res['date'].dt.day
    df_res['day_of_week'] = df_res['date'].dt.dayofweek
    df_res['is_weekend'] = (df_res['day_of_week'] >= 5).astype(int)
    df_res['is_payday'] = ((df_res['day'] == 15) | (df_res['date'].dt.is_month_end)).astype(int)
    df_res['is_month_end'] = df_res['date'].dt.is_month_end.astype(int)
    
    # Đặc trưng tuần lễ Phục Sinh (Easter Week) đặc thù
    easter_weeks = [
        ('2013-03-25', '2013-03-30'),
        ('2014-04-14', '2014-04-19'),
        ('2015-03-30', '2015-04-04'),
        ('2016-03-21', '2016-03-26'),
        ('2017-04-10', '2017-04-15'),
    ]
    df_res['is_easter_week'] = 0
    for start, end in easter_weeks:
        df_res.loc[(df_res['date'] >= start) & (df_res['date'] <= end), 'is_easter_week'] = 1
        
    # --- 5. TÍNH TOÁN BIẾN LAG VÀ ROLLING (LOGIC ĐÃ ĐỒNG BỘ MỚI 100%) ---
    print("   ↳ 6. Tính toán chuỗi biến trễ liên tiếp (Lag 1, 2, 3) và biến trượt (7, 14)...")
    # Sắp xếp đúng trục thời gian theo từng nhóm hàng để tránh tính toán sai lệch dòng
    df_res = df_res.sort_values(['family', 'date']).reset_index(drop=True)
    
    # Tạo các biến Lag liên tiếp sát ngày biên giới (16, 17, 18 ngày trước ngày hiện tại)
    for lag_name, shift_value in [('sales_lag_1', 16), ('sales_lag_2', 17), ('sales_lag_3', 18)]:
        df_res[lag_name] = df_res.groupby('family')['sales'].shift(shift_value)
        
    # Tạo biến Rolling Mean & Std bám chặt theo gốc bệ phóng shift(16)
    for window in [7, 14]:
        df_res[f'rolling_mean_{window}'] = (
            df_res.groupby('family')['sales']
            .transform(lambda x: x.shift(16).rolling(window).mean())
        )
        df_res[f'rolling_std_{window}'] = (
            df_res.groupby('family')['sales']
            .transform(lambda x: x.shift(16).rolling(window).std())
        )
        
    # --- 6. MÃ HÓA BIẾN PHÂN LOẠI (LABEL ENCODING) ---
    print("   ↳ 7. Mã hóa các biến phân loại văn bản sang số nguyên...")
    categorical_cols = ['family', 'store_type', 'city', 'state', 'holiday_national_type']
    for col in categorical_cols:
        if col in df_res.columns:
            df_res[col] = df_res[col].astype('category').cat.codes
            
    print("✅ Hoàn thành Feature Engineering nâng cao!")
    return df_res

print("🛠️ Đang tiến hành tạo các tính năng mới (Feature Engineering)...")  
# Chạy hàm xử lý tổng thể
full_featured = advanced_feature_engineering(full_data, oil_df, holidays_df, stores_df)


# ==================================================
# BƯỚC 4: TÁCH LẠI THÀNH TRAIN VÀ TEST SẠCH + LỌC FEATURE
# ==================================================
print("✂️ Đang tách dữ liệu và dọn dẹp biến thừa...")  # In thông báo phân rã bảng

# Danh sách đầy đủ đặc trưng chuẩn mong muốn được giữ lại (Đã thay thế thành lag_2, lag_3 và giữ các biến cũ)
features_to_keep = [
    'date', 'family', 'sales', 'transactions', 'is_real_holiday', 'store_type', 'cluster', 
    'is_weekend', 'is_payday', 'is_month_end', 'is_easter_week', 'day_of_week', 'year', 
    'month', 'day', 'sales_lag_1', 'sales_lag_2', 'sales_lag_3', 
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14',
    'dcoilwtico', 'city', 'state', 'holiday_national_type'  # Giữ trọn vẹn các feature cũ của bạn
]

# Tìm ngày bắt đầu nhỏ nhất của tập test (ngày 16/08/2017) để làm mốc phân chia
test_data_start = test_data['date'].min()


# --- XỬ LÝ VÀ CHỌN LỌC TẬP TEST ---
# Lọc lấy các dòng thuộc giai đoạn tương lai (từ ngày mốc test trở đi) và copy ra bảng độc lập
test_featured = full_featured[full_featured['date'] >= test_data_start].copy()

# Tạo danh sách các cột cần giữ cho tập test (loại bỏ 'sales' và 'transactions' vì tương lai chưa diễn ra)
test_cols = [col for col in features_to_keep if col in test_featured.columns and col not in ['sales', 'transactions']]

# Tiến hành ép bảng test chỉ giữ lại đúng các cột trong danh sách vừa chọn lọc ở trên
test_featured = test_featured[test_cols]


# --- XỬ LÝ VÀ CHỌN LỌC TẬP TRAIN ---
# Lọc lấy các dòng thuộc giai đoạn lịch sử (trước ngày mốc tập test) và copy ra bảng độc lập
train_featured = full_featured[full_featured['date'] < test_data_start].copy()

# Xóa bỏ các dòng của những ngày đầu năm 2013 bị trống giá trị (NaN) do cơ chế dịch chuỗi lag 16 ngày gây ra
train_featured.dropna(subset=['sales_lag_1'], inplace=True) 

# Tạo danh sách các cột cần giữ cho tập train (lọc những cột có tồn tại thực tế trong bảng full_featured)
train_cols = [col for col in features_to_keep if col in train_featured.columns]

# Tiến hành ép bảng train chỉ giữ lại đúng các cột đặc trưng mong muốn (bao gồm cả sales và transactions)
train_featured = train_featured[train_cols]


# ==================================================
# BƯỚC 5: LƯU LẠI THÀNH FILE SẴN SÀNG CHO MODEL
# ==================================================
print("💾 Đang lưu dữ liệu sạch ra file...")  # In thông báo ghi file ra ổ đĩa

# Lưu dữ liệu Train sạch ra file CSV, bỏ cột chỉ mục index thừa của pandas để giảm dung lượng file
train_featured.to_csv('train_ready_to_model.csv', index=False)

# Lưu dữ liệu Test sạch ra file CSV, bỏ cột chỉ mục index thừa của pandas phục vụ cho dự báo sau này
test_featured.to_csv('test_ready_to_model.csv', index=False)

# In số lượng dòng và cột của tập Train sau xử lý để kiểm tra tính toàn vẹn dữ liệu
print(f"👉 Shape tập Train mới (Có đầy đủ các Đặc trưng, Sales & Transactions): {train_featured.shape}")

# In số lượng dòng và cột của tập Test sau xử lý xem đã tinh gọn chuẩn xác chưa
print(f"👉 Shape tập Test mới (Tinh gọn phục vụ Predict): {test_featured.shape}")

🔄 Đang đọc dữ liệu...
🔗 Đang gộp tập dữ liệu Train và Test...
🛠️ Đang tiến hành tạo các tính năng mới (Feature Engineering)...
   ↳ 1. Sao lưu dữ liệu nền...
   ↳ 2. Ghép thông tin vị trí & phân loại cửa hàng...
   ↳ 3. Xử lý bộ lọc ngày nghỉ lễ quốc gia thực tế...
   ↳ 4. Ghép chỉ số biến động giá dầu vĩ mô...
   ↳ 5. Trích xuất đặc trưng các thành phần thời gian...
   ↳ 6. Tính toán chuỗi biến trễ liên tiếp (Lag 1, 2, 3) và biến trượt (7, 14)...
   ↳ 7. Mã hóa các biến phân loại văn bản sang số nguyên...
✅ Hoàn thành Feature Engineering nâng cao!
✂️ Đang tách dữ liệu và dọn dẹp biến thừa...
💾 Đang lưu dữ liệu sạch ra file...
👉 Shape tập Train mới (Có đầy đủ các Đặc trưng, Sales & Transactions): (3000360, 25)
👉 Shape tập Test mới (Tinh gọn phục vụ Predict): (28512, 24)


In [18]:
train = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\train_ready_to_model.csv")

test = pd.read_csv(r"D:\AIO26-Module1-Team_Level_Up-Store-Sales-Forecasting\data\test_ready_to_model.csv")


print(train.head())


         date  family  sales  is_real_holiday  store_type  cluster  \
0  2013-01-01       0    0.0                1           3        1   
1  2013-01-01       0    0.0                1           3        1   
2  2013-01-01       0    0.0                1           3       10   
3  2013-01-01       0    0.0                1           3        1   
4  2013-01-01       0    0.0                1           4       10   

   is_weekend  is_payday  is_month_end  is_easter_week  ...  sales_lag_2  \
0           0          0             0               0  ...          NaN   
1           0          0             0               0  ...          0.0   
2           0          0             0               0  ...          0.0   
3           0          0             0               0  ...          0.0   
4           0          0             0               0  ...          0.0   

   sales_lag_3  rolling_mean_7  rolling_std_7  rolling_mean_14  \
0          NaN             NaN            NaN           

In [19]:
print(train['store_type'].unique())

[3 4 2 1 0]
